### Training with **Colab**

<div style="display: flex; gap: 15px; align-items: center; padding: 15px;">
    <a href="https://colab.research.google.com/github/sadbinsiddique/Dl-net/blob/main/notebook/3.%20Train.ipynb" target="_blank">
        <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" />
    </a>
    <a href="https://github.com/settings/tokens" target="_blank" style="display: inline-flex; align-items: center; gap: 8px; padding: 8px 12px; background-color: #238636; color: white; border-radius: 6px; text-decoration: none; font-weight: 500; font-size: 14px;">
        <svg width="18" height="18" viewBox="0 0 24 24" fill="white" xmlns="http://www.w3.org/2000/svg">
            <path d="M12 0c-6.626 0-12 5.373-12 12 0 5.302 3.438 9.8 8.207 11.387.599.111.793-.261.793-.577v-2.234c-3.338.726-4.033-1.416-4.033-1.416-.546-1.387-1.333-1.756-1.333-1.756-1.089-.745.083-.729.083-.729 1.205.084 1.839 1.237 1.839 1.237 1.07 1.834 2.807 1.304 3.492.997.107-.775.418-1.305.762-1.604-2.665-.305-5.467-1.334-5.467-5.931 0-1.311.469-2.381 1.236-3.221-.124-.303-.535-1.524.117-3.176 0 0 1.008-.322 3.301 1.23.957-.266 1.983-.399 3.003-.404 1.02.005 2.047.138 3.006.404 2.291-1.552 3.297-1.23 3.297-1.23.653 1.653.242 2.874.118 3.176.77.84 1.235 1.911 1.235 3.221 0 4.609-2.807 5.624-5.479 5.921.43.372.823 1.102.823 2.222v 3.293c0 .319.192.694.801.576 4.765-1.589 8.199-6.086 8.199-11.386 0-6.627-5.373-12-12-12z"/>
        </svg>
        Access Token
    </a>
</div>

In [ ]:
from getpass import getpass
import os

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Cloud-Base Environment: {IN_COLAB}")

if IN_COLAB:
    env="colab"
    token = getpass('Enter GitHub Access Token: ')
    !git clone https://{token}@github.com/sadbinsiddique/Dl-net.git
else:
    env="local"
    print("Local environment detected. \nSkipping git clone.")

Cloud-Base Environment: False
Local environment detected. 
Skipping git clone.


In [ ]:
if IN_COLAB:
    os.chdir('/content/Dl-net')
    DATA_PATH = '/content/Dl-net/notebook/'
    print(f"Colab environment setup. Current working directory: {os.getcwd()}")
else:
    DATA_PATH = './'
    print(f"Local environment. Data path: {DATA_PATH}")

print(f"Data will be loaded from: {DATA_PATH}")

### Main Cord Start

In [16]:
import os
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from tqdm import tqdm
import pandas as pd


In [ ]:
train_df = pd.read_csv(f"{DATA_PATH}train.csv")
test_df = pd.read_csv(f"{DATA_PATH}test.csv")

test = test_df[["filepath","class"]]
train = train_df[["filepath","class"]]

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

In [18]:
test['class'].unique()

<ArrowStringArray>
['neutral', 'angry', 'sad', 'happy', 'fear']
Length: 5, dtype: str

In [19]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [20]:
from torch.utils.data import Dataset
from PIL import Image

class EmotionDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        image = Image.open(
            self.df.loc[idx,"filepath"]
        ).convert("RGB")

        label = int(self.df.loc[idx,"label"])

        if self.transform:
            image = self.transform(image)

        return image, label

In [21]:
from torch.utils.data import DataLoader

train_dataset = EmotionDataset(train, train_transform)
val_dataset = EmotionDataset(test, val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

## Alex Net


In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.alexnet(weights=models.AlexNet_Weights.DEFAULT)

num_classes = 5

model.classifier[6] = nn.Linear(
    4096,
    num_classes
)

model = model.to(device)

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-4
)

In [ ]:
from tqdm import tqdm

epochs = 20

for epoch in range(epochs):

    model.train()

    train_loss = 0
    correct = 0
    total = 0

    loop = tqdm(train_loader)

    for images, labels in loop:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

        _, predicted = outputs.max(1)

        total += labels.size(0)

        correct += predicted.eq(labels).sum().item()

        loop.set_description(f"Epoch {epoch+1}")

        loop.set_postfix(
            loss=loss.item(),
            acc=100*correct/total
        )

    print(
        f"Epoch {epoch+1} | "
        f"Loss={train_loss/len(train_loader):.4f} | "
        f"Accuracy={100*correct/total:.2f}%"
    )

  0%|          | 0/590 [00:00<?, ?it/s]